# Web scraping a book catalogue

Many useful datasets never arrive as a CSV. Sometimes the only way to get the numbers is to
read the web page a person would read. That is **web scraping**: requesting a page in code and
turning its HTML into a table.

We practise on `books.toscrape.com`, a site built expressly for this purpose. It is a sandbox,
not a real shop, so the scraping is ethical by design — and it is a good place to learn the
polite habits that apply everywhere else.

## Learning objectives

By the end of this notebook you can:

- Explain what ethical scraping means and apply its ground rules.
- Send an HTTP request with `requests`, including a user agent and a timeout.
- Parse HTML with `BeautifulSoup` using CSS selectors.
- Extract one record per item from a repeated page element.
- Follow pagination to gather several pages.
- Save the result as a tidy CSV.
- Degrade gracefully when the network is unavailable.

## Concept

**HTTP** is a request/response conversation. We send a `GET` request to a URL; the server
sends back a status code and a body. A `200` means success, `404` means the page is missing,
and `403` means we are not allowed. A `timeout` stops us waiting forever, and
`raise_for_status()` turns a bad code into an exception so failures are not silent.

**HTML** is text, but its structure is a tree of elements. Each element has a tag (`article`,
`h3`, `p`), attributes (`class="product_pod"`, `href="..."`), and children. `BeautifulSoup`
builds that tree so we can search it with **CSS selectors** such as `article.product_pod`
(every article of that class) or `p.price_color` (every price paragraph).

**Pagination** splits a long list across `page-1.html`, `page-2.html`, and so on. We loop
until a page has no items, with a safety cap so a markup change cannot cause an endless run.

**Ethics** is not optional. We send an honest user agent, sleep between requests, take only
what we need, and note the source. `books.toscrape.com` is a public sandbox that asks only for
attribution, which makes it the right place to start.

The parsing functions used here are also packaged in [`../scrape.py`](../scrape.py) as
`parse_price`, `parse_card`, and `parse_page`, so they can be imported and tested without the
network. This notebook spells them out step by step.

## Worked example

We will fetch catalogue pages, parse each book card into a row, walk the pagination, and save
a tidy CSV. Each step prints what it did, and every step after the first is guarded so that an
offline run still finishes cleanly.

### Step 1 — Fetch one page

`requests` retrieves the HTML. We name ourselves in the `User-Agent` header and set a timeout.
The whole cell is wrapped so that, with no connection, it prints a clear message instead of
crashing and leaves `html` as `None` for the later steps to check.

In [1]:
import time
from pathlib import Path

import requests
from bs4 import BeautifulSoup

from ds_practice import data_path

BASE = "http://books.toscrape.com/catalogue/page-{}.html"
HEADERS = {"User-Agent": "data-science-practice/1.0 (+portfolio scraping example)"}


def fetch_page(page: int, timeout: int = 30) -> str:
    """Return the HTML of one catalogue page, raising on a bad status."""
    response = requests.get(BASE.format(page), headers=HEADERS, timeout=timeout)
    response.raise_for_status()
    return response.text


try:
    html = fetch_page(1)
    print("fetched page 1:", len(html), "characters")
except Exception as exc:  # network down, DNS failure, timeout, bad status...
    html = None
    print("network unavailable:", exc)
    print("This notebook needs internet access; the rest of the repo does not.")
    print("You can also run: python scripts/download_data.py --module 05")

fetched page 1: 50469 characters


### Step 2 — Parse one card

Every book sits in `<article class="product_pod">`. Inside we find the title (an `h3 > a`
with a `title` attribute), the price (`p.price_color`), the rating (a `p.star-rating` whose
class list ends in a word such as `Three`), and the stock line. The rating is not text, so we
map the word to an integer ourselves. Prices use the pound sign, which we strip before
converting to a float.

In [2]:
RATING_WORDS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}


def parse_price(text: str) -> float:
    """Turn '£51.77' (or a mis-encoded variant) into 51.77."""
    cleaned = text.encode("ascii", "ignore").decode().strip()
    return float(cleaned.lstrip("£").strip())


def parse_card(card) -> dict:
    """Extract one book record from a BeautifulSoup product card."""
    title = card.h3.a["title"]
    price = parse_price(card.select_one("p.price_color").get_text(strip=True))
    classes = card.select_one("p.star-rating")["class"]
    rating = next((RATING_WORDS[c] for c in classes if c in RATING_WORDS), None)
    stock = card.select_one("p.instock.availability").get_text(strip=True)
    href = card.h3.a["href"].replace("../", "")
    return {
        "title": title,
        "price_gbp": price,
        "rating": rating,
        "in_stock": "In stock" in stock,
        "url": "http://books.toscrape.com/catalogue/" + href,
    }


if html:
    cards = BeautifulSoup(html, "html.parser").select("article.product_pod")
    print("books on page 1:", len(cards))
    for row in [parse_card(card) for card in cards[:3]]:
        print(row)
else:
    print("skipping: no HTML (offline run)")

books on page 1: 20
{'title': 'A Light in the Attic', 'price_gbp': 51.77, 'rating': 3, 'in_stock': True, 'url': 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}
{'title': 'Tipping the Velvet', 'price_gbp': 53.74, 'rating': 1, 'in_stock': True, 'url': 'http://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html'}
{'title': 'Soumission', 'price_gbp': 50.1, 'rating': 1, 'in_stock': True, 'url': 'http://books.toscrape.com/catalogue/soumission_998/index.html'}


### Step 3 — Parse a whole page

One page is just a list of cards, so a page parser is a list comprehension. Keeping this as
its own function means we can test it against a saved snippet later.

In [3]:
def parse_page(page_html: str) -> list[dict]:
    """Parse every product card on one catalogue page."""
    soup = BeautifulSoup(page_html, "html.parser")
    return [parse_card(card) for card in soup.select("article.product_pod")]


if html:
    page_rows = parse_page(html)
    print("rows from page 1:", len(page_rows))
else:
    print("skipping: no HTML (offline run)")

rows from page 1: 20


### Step 4 — Follow the pagination

We walk pages 1 to 5 with a `Session` (which reuses the connection and the headers), stopping
early if a page is missing or empty. A half-second sleep between requests keeps us polite. In
production you would also add retries and a per-host delay.

In [4]:
PAGES = 5


def scrape(pages: int = PAGES, delay: float = 0.5) -> list[dict]:
    """Collect product rows across the first ``pages`` catalogue pages."""
    rows: list[dict] = []
    session = requests.Session()
    session.headers.update(HEADERS)
    for page in range(1, pages + 1):
        response = session.get(BASE.format(page), timeout=30)
        if response.status_code != 200:
            break
        page_cards = BeautifulSoup(response.text, "html.parser").select(
            "article.product_pod"
        )
        if not page_cards:
            break
        rows.extend(parse_card(card) for card in page_cards)
        time.sleep(delay)
    return rows


books = scrape() if html else []
print("collected rows:", len(books))

collected rows: 100


### Step 5 — Save a tidy CSV

We turn the list of dictionaries into a DataFrame and write it to `data/raw/books.csv` (a
git-ignored path found through the shared helper). One row per book, one column per variable:
that is what "tidy" means. If we are offline there is nothing to save, so we say so.

In [5]:
import pandas as pd

out = data_path("books.csv")
if books:
    frame = pd.DataFrame(books).sort_values("title").reset_index(drop=True)
    out.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(out, index=False)
    print("wrote data/raw/books.csv with", len(frame), "rows")
    display(frame.head())
else:
    print("nothing to save (offline run).")
    print("Run with network access, or: python scripts/download_data.py --module 05")

wrote data/raw/books.csv with 100 rows


,title,price_gbp,rating,in_stock,url
0,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11,5,True,http://books.toscrape.com/catalogue/higherself...
1,A Light in the Attic,51.77,3,True,http://books.toscrape.com/catalogue/a-light-in...
2,Aladdin and His Wonderful Lamp,53.13,3,True,http://books.toscrape.com/catalogue/aladdin-an...
3,America's Cradle of Quarterbacks: Western Penn...,22.50,3,True,http://books.toscrape.com/catalogue/americas-c...
4,Behind Closed Doors,52.22,4,True,http://books.toscrape.com/catalogue/behind-clo...


### Step 6 — A quick sanity check

Before trusting a scrape, check that the ranges are plausible and that nothing is missing.
Prices should be positive and modest; ratings should sit between 1 and 5; there should be no
nulls.

In [6]:
if books:
    frame = pd.DataFrame(books)
    print(frame[["price_gbp", "rating"]].describe().round(2))
    print("\nmissing values per column:")
    print(frame.isna().sum())
else:
    print("skipping: no rows were collected")

       price_gbp  rating
count     100.00  100.00
mean       34.56    2.93
std        14.64    1.42
min        10.16    1.00
25%        19.90    2.00
50%        34.78    3.00
75%        47.97    4.00
max        58.11    5.00

missing values per column:
title        0
price_gbp    0
rating       0
in_stock     0
url          0
dtype: int64


## Exercises

1. **Pages 1–3.** Loop over the first three pages only and print the count per page and the
   total. Why might a page ever return fewer than 20 cards?
2. **Retry a request.** Write `fetch_with_retries(url, attempts=3)` that waits 1 second, then
   2 seconds, between attempts and re-raises the final error. Show it succeeds on page 1 and
   fails clearly on `page-9999.html`.
3. **Parse from disk.** Save page 1's HTML to `data/raw/page-1.html`, read it back, and parse
   it with `BeautifulSoup`. Confirm you get the same number of rows without touching the
   network. (Worked answers are in [`../solutions.md`](../solutions.md).)

## Limitations

The selectors are written for this particular sandbox and would need editing if the markup
changes. The scraper collects a fixed five pages and does not retry failed requests, so it
under-represents what a polite production scraper would do. Prices and ratings on the site are
fictional, so no market conclusion can be drawn from them, and we keep no history, so we
cannot see prices change over time.